In [ ]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# --- mortgage: BoE, 2yr fix. Columns: date, 95%, 90%, 75%(IUMB482), 60%, 85% ---
m = pd.read_csv("../Data/Bank of England Database.csv", skiprows=1, header=None)
m = m[[0, 3]]                                   # date + 75% LTV column
m.columns = ["date", "rate"]
m["date"] = pd.to_datetime(m["date"], format="%d %b %y", errors="coerce")
m["rate"] = pd.to_numeric(m["rate"], errors="coerce")
mort = m.dropna().sort_values("date")
mort["series"] = "2yr fixed mortgage (75% LTV)"

# --- gilt: UK 10yr from 10YT.xlsx ---
raw = pd.read_excel("../Data/10YT.xlsx")
raw = raw[~raw.iloc[:,1].astype(str).str.strip().str.lower().eq("close")]
raw = raw.rename(columns={raw.columns[0]:"date"})
raw.columns = ["date"] + [c.split(" ")[0] for c in raw.columns[1:]]
raw = raw.rename(columns={"GB10YT=RR":"United Kingdom"})
raw["date"] = pd.to_datetime(raw["date"], dayfirst=True, errors="coerce")
gilt = raw[["date","United Kingdom"]].rename(columns={"United Kingdom":"rate"})
gilt["rate"] = pd.to_numeric(gilt["rate"], errors="coerce")
gilt = gilt.dropna().set_index("date")["rate"].resample("ME").last().reset_index()
gilt["series"] = "UK 10-year gilt yield"

df = pd.concat([mort, gilt], ignore_index=True)
df = df[df.date >= "2009-01-01"]

order = ["2yr fixed mortgage (75% LTV)", "UK 10-year gilt yield"]
color = alt.Color("series:N", scale=alt.Scale(domain=order,
    range=["#E6224B", "#36B7B4"]), legend=None)

lines = alt.Chart(df).mark_line(strokeWidth=1.8).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("rate:Q", title="Interest rate",
            axis=alt.Axis(labelExpr="datum.label + '%'")),
    color=color, detail="series:N")

labels = alt.Chart(df).mark_text(align="left", dx=6, fontSize=11, fontWeight="bold").encode(
    x=alt.X("date:T", aggregate="max"),
    y=alt.Y("rate:Q", aggregate={"argmax":"date"}),
    text=alt.Text("series:N", aggregate={"argmax":"date"}), color=color)

mb = alt.Chart(pd.DataFrame({"date":pd.to_datetime(["2022-09-23"])})).mark_rule(
    color="#94a3b8", strokeDash=[2,3]).encode(x="date:T")
mb_txt = alt.Chart(pd.DataFrame({"date":pd.to_datetime(["2022-09-23"]),
    "t":["Mini-budget"]})).mark_text(align="right", dx=-4, dy=-4,
    fontSize=10, color="#64748b").encode(x="date:T", y=alt.value(12), text="t:N")

caption = alt.Title(
    text="Source: Bank of England (IUMB482), LSEG",
    subtitle=["UK 2-year fixed mortgage rate (75% LTV) and 10-year gilt yield, %. Monthly, 2009–2026."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (mb + mb_txt + lines + labels)
    .properties(width=640, height=340, padding={"right":44}, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, name="gilt_mortgage", svg=True)
chart